# R7.8 FIX1 — 六断面 × 五模型 × 三提前期预测误差公平比较（确定性基线存档修复）

本次修复：R7.6 按设计只保存 Persistence、Climatology、AR-EOF 的 Seed 71 预测数组，Seed 72 只保留复制后的误差 CSV。原 R7.8 错把这 30 个未生成的副本当成必需文件；本版先严格核对 R7.6 的两个种子逐目标日期和全部误差列完全相同，再将该三类确定性基线的 Seed 72 引用到 Seed 71 的真实预测数组。**hybrid 与 LSTM 的 Seed 71/72 均坚持独立文件，绝不互相替代。**

直接在有 D 盘数据的 WSL 内核 `Run All`，不用重训。


In [ ]:
# R7.8 FIX1 | frozen five-model six-section forecast verification | deterministic seed alias corrected | no retraining
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
import json, hashlib, io, zipfile, math, warnings, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

@dataclass
class R78Config:
    ROOT: str = '/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL: str = 'final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    R75_REL: str = 'R7_5_HybridDeepLearning_MultiYearRollingHindcast'
    R76_REL: str = 'R7_6_FairBaselines_2026'
    R75_ZIP: str = 'R75_上传给ChatGPT_多年滚动后报_hybrid_deep-learning_prediction.zip'
    R76_ZIP: str = 'R76_上传给ChatGPT_四类基线公平比较.zip'
    OUT_REL: str = 'R7_8_SixSection_FiveModels_Hindcast'
    YEARS: tuple = (2005,2008,2011,2014,2017,2019,2021,2022,2023,2024)
    SEEDS: tuple = (71,72)
    LEADS: tuple = (30,60,90)
    DROP_BRANCH1_FIRST: int = 12
    BRANCH2_MAX_LON: float = 145.0
    BOOT_REPS: int = 1500
    FIG_DPI: int = 220
    STRICT_FULL: bool = True  # require all 100 complete prediction arrays
    OUTPUT_ZIP: str = 'R78_上传给ChatGPT_六断面五模型三提前期比较.zip'

cfg = R78Config()
MODEL_KEYS = ('hybrid','persistence','climatology','ar_eof','lstm')
DETERMINISTIC_KEYS = ('persistence', 'climatology', 'ar_eof')  # R7.6 intentionally saves predictions only for the first seed
MODEL_NAMES = {'hybrid':'hybrid deep-learning prediction',
               'persistence':'Persistence', 'climatology':'Climatology',
               'ar_eof':'AR-EOF', 'lstm':'Parameter-matched LSTM'}
SECTION_LINES = {
    'A':((121.50,23.50),(125.00,22.75)),
    'B':((121.50,26.25),(124.25,24.25)),
    'PN':((126.00,29.00),(128.15,27.30)),
    'C':((130.25,31.25),(132.50,27.00)),
    'D':((137.00,34.75),(138.00,27.75)),
    'E':((141.75,40.00),(145.00,31.50)),
}
SECTION_BRANCH = {'A':'b1','B':'b2','PN':'b2','C':'b2','D':'b2','E':'b2'}
EARTH_KM = 6371.0088

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8<<20),b''):h.update(block)
    return h.hexdigest()

def read_archived(local, archive, inner, require=True):
    p=local/inner
    if p.is_file():return p.read_bytes()
    if archive.is_file():
        with zipfile.ZipFile(archive) as z:
            if inner in z.namelist():return z.read(inner)
    if require:raise FileNotFoundError(f'Missing {inner} (local {local} and archive {archive})')
    return None

def predict_path(key,year,seed):
    if key=='hybrid':
        return f'forecast_arrays/pred_{year}_seed{seed}.npz'
    return f'predictions/pred_{key}_{year}_seed{seed}.npz'

# Batch, vectorized polyline-finite section intersection, with dedup near shared vertices.
def great_circle_km(lon1,lat1,lon2,lat2):
    lon1,lat1,lon2,lat2 = map(np.asarray,(lon1,lat1,lon2,lat2))
    dlon=np.deg2rad(lon2-lon1);dlat=np.deg2rad(lat2-lat1)
    a=np.sin(dlat/2)**2+np.cos(np.deg2rad(lat1))*np.cos(np.deg2rad(lat2))*np.sin(dlon/2)**2
    return 2*EARTH_KM*np.arcsin(np.sqrt(np.clip(a,0,1)))

def intersections_batch(variable, fixed, branch, section):
    # variable shape (N,K). branch1: variable=lon, fixed=lat ; branch2: variable=lat, fixed=lon
    var=np.asarray(variable,dtype=np.float64)
    fixed=np.asarray(fixed,dtype=np.float64)
    if var.ndim!=2 or var.shape[1]!=len(fixed):raise ValueError('Invalid axes shape')
    if branch=='b1':
        x=var;y=np.broadcast_to(fixed[None,:],var.shape)
    elif branch=='b2':
        x=np.broadcast_to(fixed[None,:],var.shape);y=var
    else:raise ValueError(branch)
    (x0,y0),(x1,y1)=SECTION_LINES[section]
    rx,ry=x1-x0,y1-y0
    ax,ay=x[:,:-1],y[:,:-1]
    sx,sy=np.diff(x,axis=1),np.diff(y,axis=1)
    dx,dy=x0-ax,y0-ay
    denom=sx*ry-sy*rx
    valid_denom=(np.isfinite(denom)&(np.abs(denom)>1e-12))
    with np.errstate(divide='ignore',invalid='ignore'):
        t=np.divide(dx*ry-dy*rx,denom,where=valid_denom,out=np.full_like(denom,np.nan))
        u=np.divide(dx*sy-dy*sx,denom,where=valid_denom,out=np.full_like(denom,np.nan))
    mask=valid_denom & np.isfinite(t)&np.isfinite(u) & (t>=-1e-9)&(t<=1+1e-9)&(u>=-1e-9)&(u<=1+1e-9)
    n_raw=mask.sum(axis=1)
    n_unique=np.asarray(n_raw,dtype=np.int32).copy()
    distance=np.full(len(var),np.nan,dtype=np.float64)
    u_winners=np.full(len(var),np.nan,dtype=np.float64)
    unique_rows=np.flatnonzero(n_raw==1)
    if len(unique_rows):
        ix=mask[unique_rows].argmax(axis=1)
        u_winners[unique_rows]=np.clip(u[unique_rows,ix],0,1)
    for j in np.flatnonzero(n_raw>1):
        us=np.sort(np.clip(u[j,mask[j]],0,1))
        # 0.05 km tolerance to remove double counting at a shared vertex
        k=1
        for vprev,vnext in zip(us[:-1],us[1:]):
            if float(great_circle_km(x0+vprev*rx,y0+vprev*ry,x0+vnext*rx,y0+vnext*ry))>0.05:k+=1
        n_unique[j]=k
        if k==1:u_winners[j]=us[0]
    ok=(n_unique==1)&np.isfinite(u_winners)
    if np.any(ok):
        xx=x0+u_winners[ok]*rx;yy=y0+u_winners[ok]*ry
        distance[ok]=great_circle_km(x0,y0,xx,yy)
    return {'n_raw':n_raw.astype(np.int32),'n':n_unique, 'distance_km':distance, 'fraction_u':u_winners}

def _keyify(df):
    return list(zip(df.test_year.astype(int),df.seed.astype(int),df.issue_date.astype(str),df.target_date.astype(str),df.lead_days.astype(int)))

def run_r78(cfg=cfg):
    root=Path(cfg.ROOT);ds=root/cfg.DATASET_REL
    r75=root/cfg.R75_REL;r76=root/cfg.R76_REL
    z75=root/cfg.R75_ZIP;z76=root/cfg.R76_ZIP
    out=root/cfg.OUT_REL;tables=out/'tables';figs=out/'figures_en';reports=out/'reports'
    for d in (tables,figs,reports):d.mkdir(parents=True,exist_ok=True)
    if not ds.is_file():raise FileNotFoundError(f'GLORYS frozen truth NPZ missing: {ds}; this is required; cannot recover section errors from R76 RMSE CSV.')
    dsha=sha256(ds)
    c75=json.loads(read_archived(r75,z75,'reports/R75_冻结实验合同.json'))
    status75=json.loads(read_archived(r75,z75,'reports/R75_completed_status.json'))
    c76=json.loads(read_archived(r76,z76,'reports/R76_contract.json'))
    status76=json.loads(read_archived(r76,z76,'reports/R76_status.json'))
    assert dsha==c75['dataset_sha256']==c76['r75_dataset_sha256']==status75['dataset_sha256']==status76['dataset_sha256'],'Dataset SHA256 mismatch'
    assert tuple(status75['test_years'])==tuple(cfg.YEARS) and tuple(status75['seeds'])==tuple(cfg.SEEDS)
    assert tuple(c75['settings']['LEADS'])==tuple(cfg.LEADS)
    assert c75['settings']['DROP_BRANCH1_FIRST']==cfg.DROP_BRANCH1_FIRST
    assert c75['settings']['BRANCH2_MAX_LON']==cfg.BRANCH2_MAX_LON
    assert status75['complete'] and status76['complete']
    r75err=pd.read_csv(io.BytesIO(read_archived(r75,z75,'tables/04_all_individual_forecast_errors.csv')))
    r76err=pd.read_csv(io.BytesIO(read_archived(r76,z76,'tables/R76_all_five_model_paired_errors.csv')))
    assert len(r76err)==5*len(r75err), 'R76 all-five-model CSV rows are not 5×R75' # test counts vary by year
    r76err['model']=r76err['model'].astype(str)
    assert set(r76err['model'])==set(MODEL_NAMES.values())
    official_keys=set(_keyify(r75err))
    assert len(official_keys)==len(r75err)
    with np.load(ds,allow_pickle=False) as src:
        obs_dates=pd.to_datetime(src['dates']).strftime('%Y-%m-%d').to_numpy(dtype=str)
        raw_b1=np.asarray(src['branch1_lon'],dtype=np.float32)
        raw_lat1=np.asarray(src['branch1_target_lats'],dtype=np.float64)
        raw_b2=np.asarray(src['branch2_lat'],dtype=np.float32)
        raw_lon2=np.asarray(src['branch2_target_lons'],dtype=np.float64)
    if len(set(obs_dates.tolist()))!=len(obs_dates):raise ValueError('Ground truth duplicate dates')
    sort=np.argsort(obs_dates);obs_dates=obs_dates[sort];raw_b1=raw_b1[sort];raw_b2=raw_b2[sort]
    lat1=raw_lat1[cfg.DROP_BRANCH1_FIRST:];keep2=raw_lon2<=cfg.BRANCH2_MAX_LON+1e-8;lon2=raw_lon2[keep2]
    b1=raw_b1[:,cfg.DROP_BRANCH1_FIRST:];b2=raw_b2[:,keep2]
    index={str(d):i for i,d in enumerate(obs_dates)}
    print('DATA CONTRACT PASS',dsha[:16],'truth dates',len(obs_dates),'b1',b1.shape[1],'b2',b2.shape[1])

    # R7.6 contract: Persistence/Climatology/AR-EOF were computed ONCE per year,
    # then copied as metric rows for Seed 72. Their Seed 72 .npz files were NEVER saved.
    # Verify the two seeds' dates and all recorded errors EXACTLY agree before reusing
    # the corresponding Seed 71 prediction array (with original dataset hash audit).
    deterministic_metric_cols = ['path_mse_km2','path_rmse_km','path_mae_km','path_p90_km',
                                 'branch1_km_rmse','branch2_km_rmse']
    deterministic_checks=[]
    for model in DETERMINISTIC_KEYS:
        model_name=MODEL_NAMES[model]
        for year in cfg.YEARS:
            subset=r76err[(r76err.test_year==year)&(r76err.model==model_name)]
            by_seed=[]
            for sd in cfg.SEEDS:
                a=subset[subset.seed==sd].sort_values(['issue_date','target_date','lead_days']).reset_index(drop=True)
                by_seed.append(a)
            ref=by_seed[0]
            assert len(ref)>0, f'Deterministic baseline records missing: {model_name} {year}'
            for sd, other in zip(cfg.SEEDS[1:],by_seed[1:]):
                keycols=['issue_date','target_date','lead_days']
                if not ref[keycols].equals(other[keycols]):
                    raise RuntimeError(f'Deterministic baseline date records disagree: {model_name} {year} {cfg.SEEDS[0]} vs {sd}')
                aa=ref[deterministic_metric_cols].to_numpy(dtype=float)
                bb=other[deterministic_metric_cols].to_numpy(dtype=float)
                if not np.array_equal(aa,bb, equal_nan=True):
                    raise RuntimeError(f'Deterministic baseline predictions ARE NOT identical in official R7.6 metrics: {model_name} {year}, seed {sd}; refuse alias')
            deterministic_checks.append(dict(test_year=year,model=model_name,
                n_issues=ref.issue_date.nunique(),n_lead_rows=len(ref),
                metric_rows_identical_across_seeds=True))
    pd.DataFrame(deterministic_checks).to_csv(tables/'00b_deterministic_seed_equivalence.csv',index=False,encoding='utf-8-sig')
    print('[R76 DETERMINISTIC AUDIT PASS]',len(deterministic_checks),'year/model cases:',
          'official Seed 71/72 dates and all six error metrics are exactly identical')

    # Preflight: 20 hybrid and 20 LSTM predictions MUST be independent by seed;
    # 3 deterministic baselines × 10 years only need one verified array per year.
    # Strict mode still prevents any genuinely unavailable source, rather than
    # treating deliberately unmaterialized seed copies as missing files.
    expected=[];unavailable=[];source_lookup={}
    archive_member_cache={}
    def source_exists(model, inner):
        sub=(r75 if model=='hybrid' else r76);arc=(z75 if model=='hybrid' else z76)
        local=(sub/inner).is_file()
        inzip=False
        if not local and arc.is_file():
            if arc not in archive_member_cache:
                with zipfile.ZipFile(arc) as zz: archive_member_cache[arc]=set(zz.namelist())
            inzip=(inner in archive_member_cache[arc])
        return local,inzip
    for year in cfg.YEARS:
        for seed in cfg.SEEDS:
            for model in MODEL_KEYS:
                requested_inner=predict_path(model,year,seed)
                existing,inzip=source_exists(model,requested_inner)
                source_inner=requested_inner
                source_seed=seed
                intentional_seed_reuse=False
                if not (existing or inzip) and model in DETERMINISTIC_KEYS and seed != cfg.SEEDS[0]:
                    # Justified by the zero-difference R76 CSV audit above; no fake re-training.
                    source_seed=cfg.SEEDS[0]
                    source_inner=predict_path(model,year,source_seed)
                    existing,inzip=source_exists(model,source_inner)
                    intentional_seed_reuse=bool(existing or inzip)
                record={'test_year':year,'seed':seed,'model':MODEL_NAMES[model],
                        'requested_file':requested_inner,'source_file':source_inner,
                        'source_seed':int(source_seed),
                        'deterministic_seed_reuse':intentional_seed_reuse,
                        'local_exists':existing,'zip_exists':inzip,'available':existing or inzip}
                expected.append(record)
                source_lookup[(year,seed,model)]=source_inner
                if not record['available']:unavailable.append(record)
    audit=pd.DataFrame(expected)
    audit.to_csv(tables/'00_prediction_availability_audit.csv',index=False,encoding='utf-8-sig')
    reused=int(audit.deterministic_seed_reuse.sum())
    print(f'[R78 INPUT] prediction requests: {len(expected)}, deterministic Seed-71 reuse: {reused}, genuine missing: {len(unavailable)}')
    if unavailable:
        (reports/'MISSING_PREDICTIONS.txt').write_text('Missing genuinely required source prediction arrays (not intentional deterministic seed copies).\n'+
            '\n'.join(f"{x['model']} {x['test_year']} seed{x['seed']}: {x['source_file']}" for x in unavailable)+'\n',encoding='utf8')
        print(f'[INPUT WARNING] {len(unavailable)}/100 REQUIRED SOURCES missing. First: {unavailable[0]}')
        if cfg.STRICT_FULL:
            raise FileNotFoundError('Some REQUIRED prediction arrays are genuinely missing. R7.6 deterministic seed duplicates have already been resolved automatically; check tables/00_prediction_availability_audit.csv for remaining missing sources.')
    print(f'Predictions available {len(expected)-len(unavailable)}/{len(expected)} (including audited deterministic reuse). No training will run.')

    # Cache truth intersections by actual target date, separate from model predictions.
    all_targets=np.unique(r75err.target_date.astype(str).to_numpy())
    missing_truth_dates=[dt for dt in all_targets if dt not in index]
    if missing_truth_dates:raise RuntimeError(f'Official R75 target date absent from truth: {missing_truth_dates[:5]}')
    target_to_pos={d:i for i,d in enumerate(all_targets)}
    target_idx=np.array([index[d] for d in all_targets],dtype=int)
    truth_sections={}
    for section,branch in SECTION_BRANCH.items():
        true_var=b1[target_idx] if branch=='b1' else b2[target_idx]
        fixed=lat1 if branch=='b1' else lon2
        truth_sections[section]=intersections_batch(true_var,fixed,branch,section)
    # Second source only for overlap diagnostic of section B
    b_truth_b1=intersections_batch(b1[target_idx],lat1,'b1','B')
    n_total=0;all_parts=[];b_sens=[]
    for year in cfg.YEARS:
        for seed in cfg.SEEDS:
            official=r75err[(r75err.test_year==year)&(r75err.seed==seed)]
            official=official[official.lead_days==cfg.LEADS[0]]
            expected_issues=official.issue_date.astype(str).to_numpy()
            expected_targets={lead:r75err[(r75err.test_year==year)&(r75err.seed==seed)&(r75err.lead_days==lead)]
                                .target_date.astype(str).to_numpy() for lead in cfg.LEADS}
            for model in MODEL_KEYS:
                sub=(r75 if model=='hybrid' else r76);arc=(z75 if model=='hybrid' else z76)
                inner=source_lookup[(year,seed,model)]
                raw=read_archived(sub,arc,inner,require=False)
                if raw is None:continue
                with np.load(io.BytesIO(raw),allow_pickle=False) as p:
                    pp1=np.asarray(p['pred_b1_lon']);pp2=np.asarray(p['pred_b2_lat']);issues=np.asarray(p['issue_date']).astype(str)
                    tg=np.asarray(p['target_date']).astype(str);ll=np.asarray(p['lead_days'],int)
                    hashp=str(p['dataset_sha256'].item());name=str(p['model'].item())
                    if 'branch1_fixed_lats' in p.files and not np.allclose(p['branch1_fixed_lats'],lat1,atol=1e-9):raise RuntimeError('b1 lat grid mismatched')
                    if 'branch2_fixed_lons' in p.files and not np.allclose(p['branch2_fixed_lons'],lon2,atol=1e-9):raise RuntimeError('b2 lon grid mismatched')
                if hashp!=dsha or name!=MODEL_NAMES[model]:raise RuntimeError(f'Wrong CID/model/hash {inner}: {hashp},{name}')
                if not np.array_equal(ll,np.asarray(cfg.LEADS)) or not np.array_equal(issues,expected_issues):raise RuntimeError(f'Issue dates incorrect: {inner}')
                if pp1.shape!=(len(issues),len(ll),len(lat1)) or pp2.shape!=(len(issues),len(ll),len(lon2)):
                    raise RuntimeError(f'Forecast tensor incorrect: {inner}: {pp1.shape} {pp2.shape}')
                if not (np.isfinite(pp1).all() and np.isfinite(pp2).all()):raise RuntimeError('nonfinite predictions '+inner)
                for k,lead in enumerate(ll):
                    if not np.array_equal(tg[:,k],expected_targets[int(lead)]):raise RuntimeError(f'Target date not equal to R75 truth contract: {inner}+{lead}')
                    dcheck=(pd.to_datetime(tg[:,k])-pd.to_datetime(issues)).days.to_numpy()
                    if not (dcheck==int(lead)).all():raise RuntimeError(f'Temporal shift mismatch: {inner}+{lead}')
                nm=len(issues)*len(ll)
                datesflat=tg.reshape(-1);issueflat=np.repeat(issues,len(ll));leadflat=np.tile(ll,len(issues))
                tidx=np.array([target_to_pos[x] for x in datesflat],dtype=np.int32)
                refmask_keys={(year,seed,issues[j],str(tg[j,k]),int(ll[k])) for j in range(len(issues)) for k in range(len(ll))}
                if refmask_keys!={(year,seed,*x[2:]) for x in official_keys if x[0]==year and x[1]==seed}:
                    raise RuntimeError(f'Forecast manifest differs from official R75 dates: {inner}')
                for section,branch in SECTION_BRANCH.items():
                    pred_var=(pp1.reshape(nm,-1) if branch=='b1' else pp2.reshape(nm,-1))
                    pred_ix=intersections_batch(pred_var,lat1 if branch=='b1' else lon2,branch,section)
                    truth_ix=truth_sections[section]
                    nt=truth_ix['n'][tidx];npred=pred_ix['n']
                    valid=(nt==1)&(npred==1)
                    truth_d=truth_ix['distance_km'][tidx];pred_d=pred_ix['distance_km']
                    err=np.full(nm,np.nan,dtype=float);err[valid]=pred_d[valid]-truth_d[valid]
                    df=pd.DataFrame({'test_year':year,'seed':seed,'model':MODEL_NAMES[model],'section':section,
                        'branch':branch,'issue_date':issueflat,'target_date':datesflat,'lead_days':leadflat,
                        'truth_n_intersections':nt,'forecast_n_intersections':npred,
                        'truth_raw_intersections':truth_ix['n_raw'][tidx],
                        'forecast_raw_intersections':pred_ix['n_raw'],
                        'truth_distance_km':truth_d,'forecast_distance_km':pred_d,
                        'signed_error_km':err,'abs_error_km':np.abs(err),
                        'squared_error_km2':err**2,'paired_valid':valid,
                        'reason':np.where(nt!=1,'reference_not_unique',np.where(npred!=1,'forecast_not_unique','ok'))})
                    all_parts.append(df)
                # Prespecified section-B branch1 sensitivity, not mixed with primary B2 table
                bisect=intersections_batch(pp1.reshape(nm,-1),lat1,'b1','B')
                q=b_truth_b1
                trueb=q['distance_km'][tidx];predb=bisect['distance_km']
                ok=(bisect['n']==1)&(q['n'][tidx]==1)
                e=np.full(nm,np.nan);e[ok]=predb[ok]-trueb[ok]
                b_sens.append(pd.DataFrame({'test_year':year,'seed':seed,'model':MODEL_NAMES[model],
                    'section':'B_branch1_sensitivity','lead_days':leadflat,'issue_date':issueflat,
                    'target_date':datesflat,'paired_valid':ok,'signed_error_km':e,'squared_error_km2':e**2}))
                n_total+=nm
            print(f'[{year} seed {seed}] scored present models; total analyzed model-target pairs {n_total:,}')
    if not all_parts:raise RuntimeError('No predictions scored')
    detail=pd.concat(all_parts,ignore_index=True)
    detail['test_year']=detail.test_year.astype(int)
    detail['seed']=detail.seed.astype(int)
    # prediction-by-section main auditable table, includes all failures and denominator
    detail.to_csv(tables/'01_all_model_section_forecasts_with_QC.csv',index=False,encoding='utf-8-sig')
    keys=['test_year','seed','issue_date','target_date','lead_days','section']
    if detail[keys+['model']].duplicated().any():raise RuntimeError('Duplicate predicted section records')
    # Complete-case five-model pairing over identical dates: no wins due to differing intersection coverage
    vc=detail.pivot(index=keys,columns='model',values='paired_valid')
    present=[MODEL_NAMES[m] for m in MODEL_KEYS]
    if all(k in vc.columns for k in present):
        common=vc[present].fillna(False).all(axis=1)
        eligible=vc.index[common].to_frame(index=False)
        eligible['fair_paired_five_models']=True
        fair=detail.merge(eligible,on=keys,how='inner',validate='many_to_one')
    else:
        fair=detail.iloc[:0].copy()
    fair.to_csv(tables/'02_fair_common_date_section_errors.csv',index=False,encoding='utf-8-sig')
    coverage=detail.groupby(['section','model','lead_days']).agg(
        n_total=('paired_valid','size'),n_valid=('paired_valid','sum'),
        n_ref_unique=('truth_n_intersections',lambda x:int((x==1).sum())),
        n_forecast_unique=('forecast_n_intersections',lambda x:int((x==1).sum())),
        n_ref_missing=('truth_n_intersections',lambda x:int((x==0).sum())),
        n_forecast_missing=('forecast_n_intersections',lambda x:int((x==0).sum())),
        n_forecast_multicross=('forecast_n_intersections',lambda x:int((x>1).sum()))).reset_index()
    coverage['coverage_pct']=coverage['n_valid']/coverage['n_total']*100
    coverage.to_csv(tables/'03_native_coverage_all_predictions.csv',index=False,encoding='utf-8-sig')

    def summaries(df):
        if len(df)==0:return pd.DataFrame()
        z=df[df['paired_valid']].copy()
        if len(z)==0:return pd.DataFrame()
        grouped=z.groupby(['section','model','lead_days'],sort=False)
        native=grouped.agg(n_valid=('signed_error_km','size'),MSE_km2=('squared_error_km2','mean'),
                MAE_km=('abs_error_km','mean'),bias_km=('signed_error_km','mean'),
                medAE_km=('abs_error_km','median'),P90AE_km=('abs_error_km',lambda q:float(np.percentile(q,90))),
                years=('test_year','nunique')).reset_index()
        native['RMSE_km']=np.sqrt(native['MSE_km2'])
        ym=z.groupby(['section','model','lead_days','test_year']).agg(
            n_valid=('signed_error_km','size'),MSE_km2=('squared_error_km2','mean'),
            MAE_km=('abs_error_km','mean'),bias_km=('signed_error_km','mean')).reset_index()
        # macro-year equal MSE first, then sqrt; paired fair always same year/date for all methods
        macro=ym.groupby(['section','model','lead_days']).agg(
            n_years=('test_year','nunique'),year_equal_MSE_km2=('MSE_km2','mean'),
            year_equal_MAE_km=('MAE_km','mean'),year_equal_bias_km=('bias_km','mean')).reset_index()
        macro['year_equal_RMSE_km']=np.sqrt(macro['year_equal_MSE_km2'])
        return native,ym,macro
    native_sum, native_year, native_macro=summaries(detail)
    fair_sum,fair_year,fair_macro = summaries(fair) if len(fair) else (pd.DataFrame(),pd.DataFrame(),pd.DataFrame())
    native_sum.to_csv(tables/'04_native_RMSE_MAE_bias.csv',index=False,encoding='utf-8-sig')
    native_year.to_csv(tables/'05_native_year_by_model_section_lead.csv',index=False,encoding='utf-8-sig')
    native_macro.to_csv(tables/'06_native_macro_by_model_section_lead.csv',index=False,encoding='utf-8-sig')
    fair_sum.to_csv(tables/'07_fair_common_cases_pooled_metrics.csv',index=False,encoding='utf-8-sig')
    fair_year.to_csv(tables/'08_fair_year_by_model_section_lead.csv',index=False,encoding='utf-8-sig')
    fair_macro.to_csv(tables/'09_MAIN_fair_year_equal_90_metrics.csv',index=False,encoding='utf-8-sig')
    if len(fair_macro):
        piv=fair_macro.pivot(index=['section','model'],columns='lead_days',values='year_equal_RMSE_km').reset_index()
        piv.columns=[f'RMSE_{v}d_km' if isinstance(v,(int,np.integer)) else v for v in piv.columns]
        secord={s:i for i,s in enumerate(SECTION_LINES)};mord={MODEL_NAMES[m]:i for i,m in enumerate(MODEL_KEYS)}
        piv=piv.sort_values(['section','model'],key=lambda x: x.map(secord if x.name=='section' else mord))
        piv.to_csv(tables/'10_MAIN_six_sections_five_models_three_leads_pivot.csv',index=False,encoding='utf-8-sig')
    else:piv=pd.DataFrame()
    common_count=fair.groupby(['section','lead_days']).size().div(5).reset_index(name='n_common_issue_seed_cases') if len(fair) else pd.DataFrame()
    if len(common_count):common_count.to_csv(tables/'11_common_date_denominator.csv',index=False,encoding='utf-8-sig')

    # Comparison CIs by resampling years, paired across all five models, with same sample set.
    ci=[]
    if len(fair_year):
        rng=np.random.default_rng(20261020)
        for section in SECTION_LINES:
            for lead in cfg.LEADS:
                sub=fair_year[(fair_year.section==section)&(fair_year.lead_days==lead)]
                mat=sub.pivot(index='test_year',columns='model',values='MSE_km2')
                years=mat.index.to_numpy(dtype=int)
                if len(years)<3 or mat.isna().any().any():
                    warnings.warn(f'CI skipped due to sparse years {section}+{lead}');continue
                arr=mat[present].to_numpy(float)
                rep=rng.integers(0,len(years),size=(cfg.BOOT_REPS,len(years)))
                boot=np.sqrt(arr[rep].mean(axis=1))
                for j,mod in enumerate(present):
                    lo,hi=np.percentile(boot[:,j],[2.5,97.5])
                    ci.append({'section':section,'lead_days':lead,'comparison':mod,
                       'metric':'model_RMSE_km','point':float(np.sqrt(arr[:,j].mean())),
                       'ci_low':float(lo),'ci_high':float(hi),'n_years':len(years)})
                for j,mod in enumerate(present[1:],1):
                    delta=boot[:,j]-boot[:,0] # positive baseline - hybrid -> hybrid better
                    lo,hi=np.percentile(delta,[2.5,97.5])
                    ci.append({'section':section,'lead_days':lead,'comparison':mod+' minus hybrid',
                      'metric':'paired_RMSE_delta_km',
                      'point':float(np.sqrt(arr[:,j].mean())-np.sqrt(arr[:,0].mean())),
                      'ci_low':float(lo),'ci_high':float(hi),'n_years':len(years)})
    pd.DataFrame(ci).to_csv(tables/'12_year_block_bootstrap_95CI.csv',index=False,encoding='utf-8-sig')
    bs=pd.concat(b_sens,ignore_index=True)
    bvalid=bs[bs['paired_valid']].copy()
    if len(bvalid):
        bb=bvalid.groupby(['model','lead_days']).agg(n=('paired_valid','size'),mse=('squared_error_km2','mean')).reset_index()
        bb['RMSE_km']=np.sqrt(bb['mse'])
        bb.to_csv(tables/'13_section_B_branch1_sensitivity.csv',index=False,encoding='utf-8-sig')

    # paper-quality PNG/PDF: heatmap per lead, metric and QC in separate figures
    def heatmap(mat,labels,xlabels,title,name,fmt='.1f',unit='km'):
        fig,ax=plt.subplots(figsize=(10.5,4.8),constrained_layout=True)
        masked=np.ma.masked_invalid(mat)
        im=ax.imshow(masked,aspect='auto',cmap='viridis')
        ax.set_yticks(np.arange(len(labels)),labels);ax.set_xticks(np.arange(len(xlabels)),xlabels,rotation=18,ha='right')
        ax.set_title(title,fontsize=11,pad=14)
        plt.colorbar(im,ax=ax,shrink=.85,label=unit)
        for r in range(mat.shape[0]):
            for col in range(mat.shape[1]):
                if np.isfinite(mat[r,col]):
                    ax.text(col,r,format(mat[r,col],fmt),ha='center',va='center',fontsize=9,
                            color=('black' if np.max(masked)-np.min(masked)<1e-9 or mat[r,col]>(np.nanmin(mat)+np.nanmax(mat))/2 else 'white'))
        for suff in ['png','pdf']:
            fig.savefig(figs/f'{name}.{suff}',dpi=cfg.FIG_DPI,bbox_inches='tight')
        plt.show();plt.close(fig)
    if len(fair_macro):
        labels=list(SECTION_LINES);xlabels=list(present)
        for lead in cfg.LEADS:
            d0=fair_macro[fair_macro.lead_days==lead]
            mat=d0.set_index(['section','model']).year_equal_RMSE_km.reindex(pd.MultiIndex.from_product([labels,xlabels])).to_numpy().reshape(6,5)
            heatmap(mat,labels,xlabels,f'{lead}-day section-intersection RMSE | matched dates, year-equal MSE',
                    f'Fig_R78_{lead}d_five_models_section_RMSE',unit='RMSE (km)')
        for lead in cfg.LEADS:
            d0=coverage[coverage.lead_days==lead]
            mat=d0.set_index(['section','model']).coverage_pct.reindex(pd.MultiIndex.from_product([labels,xlabels])).to_numpy().reshape(6,5)
            heatmap(mat,labels,xlabels,f'{lead}-day section crossing coverage | per-model native',
                    f'Fig_R78_{lead}d_crossing_coverage',fmt='.0f',unit='valid (%)')

    report={'stage':'completed' if not unavailable and len(fair_macro)==90 else 'partial',
         'created_utc':datetime.now(timezone.utc).isoformat(),'dataset_sha256':dsha,
         'config':asdict(cfg),'sections':SECTION_LINES,'branch_policy':SECTION_BRANCH,
         'model_keys':MODEL_NAMES,'n_model_forecast_lead_records':int(n_total),
         'n_predicted_section_records':int(len(detail)),
         'n_five_model_common_valid_section_records':int(len(fair)),
         'n_common_cases_divide5':int(len(fair)//5),
         'missing_prediction_files':len(unavailable),
         'deterministic_seed_reuse_count':int(audit.deterministic_seed_reuse.sum()),
         'deterministic_seed_reuse_explanation':'R7.6 saved only Seed 71 forecast arrays for Persistence/Climatology/AR-EOF; Seed 72 official date keys and all six scoring columns were verified exactly identical before reuse. Reuse is NOT an independent seed replicate.',
         'reference':'GLORYS12V1 50m frozen dual-branch paths, NOT satellite ADT',
         'strict_comparison':'A=branch1, B/PN/C/D/E=branch2; one unique crossing on both truth and forecast',
         'primary_aggregation':'within-year seed/issue MSE, across years equal-weight MSE, then sqrt',
         'no_training_or_checkpoint_loading':True,
         'main_tables':['09_MAIN_fair_year_equal_90_metrics.csv','10_MAIN_six_sections_five_models_three_leads_pivot.csv'],
         'limitations':['Predicted/nonpredicted section intersections can differ; coverage reported',
                        'Ten selected annual folds, not exhaustive 20-year rolling tests',
                        '2011,2014,2019 previously used in development','B overlap sensitivity reported separately']}
    (reports/'R78_run_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf8')
    zpath=root/cfg.OUTPUT_ZIP;tmp=Path(str(zpath)+'.part')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
        for directory in (tables,reports,figs):
            for p in sorted(directory.glob('*')):
                if p.is_file():z.write(p,arcname=f'{directory.name}/{p.name}')
    tmp.replace(zpath)
    with zipfile.ZipFile(zpath) as z:assert z.testzip() is None
    digest=sha256(zpath)
    Path(str(zpath)+'.sha256').write_text(f'{digest}  {zpath.name}\n',encoding='utf8')
    print('\\n=== R78 COMPLETE ===',report['stage'])
    print('Main 90 metrics:',tables/'09_MAIN_fair_year_equal_90_metrics.csv')
    print('Six by five pivot:',tables/'10_MAIN_six_sections_five_models_three_leads_pivot.csv')
    print('Upload:',zpath,'sha256',digest)
    if len(piv):
        try:
            from IPython.display import display as ip_display
            ip_display(piv.round(2))
        except Exception:print(piv.round(2).to_string(index=False))
    try:
        from IPython.display import display as ip_display, FileLink
        ip_display(FileLink(str(zpath)));ip_display(FileLink(str(zpath)+'.sha256'))
    except Exception:pass
    return report



In [ ]:
# R7.8 FIX1 | 读取本地冻结数据，自动校验确定性基线 Seed 71/72 等价性，再进行五模型六断面统计
result = run_r78(cfg)


In [ ]:
# ===== R78 绘制：6条断面 × 5模型 × 30/60/90天 折线图 =====
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

# -----------------------------
# 1. 路径配置
# -----------------------------
ROOT = Path("/mnt/d/KuroshioDaily1993_2025")
OUT = ROOT / "R78_六断面五模型三提前期比较"
TABLE_DIR = OUT / "tables"
FIG_EN = OUT / "figures_en_line"
FIG_ZH = OUT / "figures_zh_line"
FIG_EN.mkdir(parents=True, exist_ok=True)
FIG_ZH.mkdir(parents=True, exist_ok=True)

# 优先读取正式主表；如果不存在，则读取 pivot 表
csv_main = TABLE_DIR / "09_MAIN_fair_year_equal_90_metrics.csv"
csv_pivot = TABLE_DIR / "10_MAIN_six_sections_five_models_three_leads_pivot.csv"

if csv_main.exists():
    df = pd.read_csv(csv_main)
else:
    df = pd.read_csv(csv_pivot)

print("Loaded:", csv_main if csv_main.exists() else csv_pivot)
print(df.head())

# -----------------------------
# 2. 兼容字段名
# -----------------------------
def find_col(candidates, columns):
    for c in candidates:
        if c in columns:
            return c
    return None

cols = list(df.columns)

col_section = find_col(["section", "Section", "断面"], cols)
col_model   = find_col(["model", "Model", "模型"], cols)
col_lead    = find_col(["lead_days", "LeadDays", "提前期_天", "lead"], cols)
col_rmse    = find_col(["rmse_km", "RMSE_km", "RMSE", "预测误差RMSE_km"], cols)

if any(x is None for x in [col_section, col_model, col_lead, col_rmse]):
    raise ValueError(
        f"无法识别字段，请检查 CSV 列名。\n当前列名：{cols}\n"
        "至少需要断面、模型、提前期、RMSE这4类字段。"
    )

plot_df = df[[col_section, col_model, col_lead, col_rmse]].copy()
plot_df.columns = ["section", "model", "lead_days", "rmse_km"]

# 如果 lead_days 不是数字，尝试转换
plot_df["lead_days"] = pd.to_numeric(plot_df["lead_days"], errors="coerce")
plot_df["rmse_km"] = pd.to_numeric(plot_df["rmse_km"], errors="coerce")
plot_df = plot_df.dropna(subset=["lead_days", "rmse_km"])

# 只保留 30 / 60 / 90
plot_df = plot_df[plot_df["lead_days"].isin([30, 60, 90])].copy()

# -----------------------------
# 3. 模型顺序与断面顺序
# -----------------------------
section_order = ["A", "B", "PN", "C", "D", "E"]

preferred_model_order = [
    "hybrid deep-learning prediction",
    "Persistence",
    "Climatology",
    "AR-EOF",
    "Parameter-matched LSTM",
]

models_present = plot_df["model"].dropna().unique().tolist()
model_order = [m for m in preferred_model_order if m in models_present]
for m in models_present:
    if m not in model_order:
        model_order.append(m)

plot_df["section"] = pd.Categorical(plot_df["section"], categories=section_order, ordered=True)
plot_df["model"] = pd.Categorical(plot_df["model"], categories=model_order, ordered=True)
plot_df = plot_df.sort_values(["section", "model", "lead_days"])

print("Sections:", plot_df["section"].dropna().unique().tolist())
print("Models:", model_order)

# -----------------------------
# 4. 中文字体设置（避免乱码）
# -----------------------------
def pick_chinese_font():
    candidates = [
        "Noto Sans CJK SC", "Noto Sans CJK JP", "Microsoft YaHei",
        "SimHei", "WenQuanYi Zen Hei", "Arial Unicode MS"
    ]
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            return name
    return None

zh_font = pick_chinese_font()
if zh_font is not None:
    plt.rcParams["font.sans-serif"] = [zh_font]
plt.rcParams["axes.unicode_minus"] = False

# -----------------------------
# 5. 每条断面单独画一张英文图
# -----------------------------
for sec in section_order:
    sub = plot_df[plot_df["section"] == sec].copy()
    if len(sub) == 0:
        continue

    plt.figure(figsize=(7.2, 5.0))
    for model in model_order:
        d = sub[sub["model"] == model].sort_values("lead_days")
        if len(d) == 0:
            continue
        plt.plot(
            d["lead_days"], d["rmse_km"],
            marker="o", linewidth=2, markersize=6,
            label=model
        )

    plt.xticks([30, 60, 90], ["30 days", "60 days", "90 days"])
    plt.xlabel("Lead time")
    plt.ylabel("Prediction RMSE (km)")
    plt.title(f"Section {sec}: prediction RMSE of five models")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_EN / f"Section_{sec}_five_models_line_EN.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIG_EN / f"Section_{sec}_five_models_line_EN.pdf", bbox_inches="tight")
    plt.close()

# -----------------------------
# 6. 每条断面单独画一张中文图
# -----------------------------
for sec in section_order:
    sub = plot_df[plot_df["section"] == sec].copy()
    if len(sub) == 0:
        continue

    plt.figure(figsize=(7.2, 5.0))
    for model in model_order:
        d = sub[sub["model"] == model].sort_values("lead_days")
        if len(d) == 0:
            continue
        plt.plot(
            d["lead_days"], d["rmse_km"],
            marker="o", linewidth=2, markersize=6,
            label=model
        )

    plt.xticks([30, 60, 90], ["30天", "60天", "90天"])
    plt.xlabel("预测提前期")
    plt.ylabel("预测误差 RMSE (km)")
    plt.title(f"{sec}断面：5个模型在不同提前期的预测误差")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_ZH / f"{sec}断面_五模型折线图_ZH.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIG_ZH / f"{sec}断面_五模型折线图_ZH.pdf", bbox_inches="tight")
    plt.close()

# -----------------------------
# 7. 额外生成一个 6 联图（英文）
# -----------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=False)
axes = axes.flatten()

for i, sec in enumerate(section_order):
    ax = axes[i]
    sub = plot_df[plot_df["section"] == sec].copy()
    for model in model_order:
        d = sub[sub["model"] == model].sort_values("lead_days")
        if len(d) == 0:
            continue
        ax.plot(
            d["lead_days"], d["rmse_km"],
            marker="o", linewidth=2, markersize=5,
            label=model
        )
    ax.set_title(f"Section {sec}")
    ax.set_xticks([30, 60, 90])
    ax.set_xticklabels(["30", "60", "90"])
    ax.set_xlabel("Lead time (days)")
    ax.set_ylabel("RMSE (km)")
    ax.grid(True, alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, fontsize=9, frameon=False)
fig.suptitle("Prediction RMSE of five models on six sections", y=0.98, fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(FIG_EN / "Six_sections_five_models_line_panels_EN.png", dpi=300, bbox_inches="tight")
plt.savefig(FIG_EN / "Six_sections_five_models_line_panels_EN.pdf", bbox_inches="tight")
plt.close()

# -----------------------------
# 8. 额外生成一个 6 联图（中文）
# -----------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=False)
axes = axes.flatten()

for i, sec in enumerate(section_order):
    ax = axes[i]
    sub = plot_df[plot_df["section"] == sec].copy()
    for model in model_order:
        d = sub[sub["model"] == model].sort_values("lead_days")
        if len(d) == 0:
            continue
        ax.plot(
            d["lead_days"], d["rmse_km"],
            marker="o", linewidth=2, markersize=5,
            label=model
        )
    ax.set_title(f"{sec}断面")
    ax.set_xticks([30, 60, 90])
    ax.set_xticklabels(["30天", "60天", "90天"])
    ax.set_xlabel("预测提前期")
    ax.set_ylabel("RMSE (km)")
    ax.grid(True, alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, fontsize=9, frameon=False)
fig.suptitle("六条断面上五个模型在不同提前期的预测误差", y=0.98, fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(FIG_ZH / "六断面_五模型_折线图联图_ZH.png", dpi=300, bbox_inches="tight")
plt.savefig(FIG_ZH / "六断面_五模型_折线图联图_ZH.pdf", bbox_inches="tight")
plt.close()

print("Done.")
print("English figures saved to:", FIG_EN)
print("Chinese figures saved to:", FIG_ZH)

## 输出说明

`tables/00_prediction_availability_audit.csv`：记录全部 100 个模型×年份×Seed 请求及真实 `source_seed`、`source_file`，预期有 30 个 `deterministic_seed_reuse=True`，没有真正缺失的源文件。

`tables/00b_deterministic_seed_equivalence.csv`：逐年份、逐基线检查官方 R7.6 两 Seed 的日期和六个误差数值完全一致。

`tables/09_MAIN_fair_year_equal_90_metrics.csv` 和 `tables/10_MAIN_six_sections_five_models_three_leads_pivot.csv`：论文分析用完整 90 项主表，所有模型采用相同交点有效日期。

确定性基线的 Seed 72 预测只是 Seed 71 真实数组的正确重复使用；它不是独立训练运行。强制 `STRICT_FULL=True` 保留。
